# 03 — LoRA SFT Training

Train LoRA adapters on filtered SynSQL data using Unsloth + TRL.
Grid search over rank, target modules, learning rate, RSLoRA.

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/SLM-PRJ')
os.chdir('/kaggle/working/SLM-PRJ')

from training.lora_config import generate_experiments
from training.sft_trainer import train, load_sft_dataset
from training.merge_adapter import merge_and_save
import json, time
from pathlib import Path

## Grid Search on 5K Subset

In [ ]:
experiments = generate_experiments('configs/lora_grid.yaml', 'qwen2.5-coder-0.5b')
print(f'{len(experiments)} experiments to run')
for e in experiments:
    print(f'  {e.name}: rank={e.rank}, modules={e.target_modules}, lr={e.lr}, rslora={e.rslora}')

In [ ]:
TRAIN_DATA = 'data/synsql_filtered_10000.jsonl'  # 5-10K subset
RESULTS_DIR = Path('results/lora')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

grid_results = []
for exp in experiments:
    print(f'\n=== Training {exp.name} ===')
    start = time.time()
    try:
        result = train(exp, TRAIN_DATA, str(RESULTS_DIR / exp.name))
        result['wall_time_s'] = round(time.time() - start, 1)
        grid_results.append({'name': exp.name, **result})
        print(f'  Done in {result["wall_time_s"]}s, loss={result.get("final_loss", "?")}')
    except Exception as e:
        print(f'  FAILED: {e}')
        grid_results.append({'name': exp.name, 'error': str(e)})

with open(RESULTS_DIR / 'grid_results.json', 'w') as f:
    json.dump(grid_results, f, indent=2)
print('\nGrid search complete.')

## Quick Eval on 300 Dev Questions

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_db_path, get_schema
from benchmarks.eval_ex import execution_match
from vllm import LLM, SamplingParams
from training.prompt_template import SYSTEM_MESSAGE
from core.prompt_builder import build_prompt

DATA_DIR = Path('/kaggle/input/bird-benchmark')
bird_dev = load_bird_dev(DATA_DIR)[:300]

best_adapters = [r for r in grid_results if 'error' not in r]
best_adapters.sort(key=lambda r: r.get('final_loss', 999))
top_adapter = best_adapters[0]['name'] if best_adapters else None
print(f'Evaluating best adapter: {top_adapter}')

In [ ]:
if top_adapter:
    adapter_path = str(RESULTS_DIR / top_adapter / 'final')
    merge_path = str(RESULTS_DIR / top_adapter / 'merged')
    merge_and_save('Qwen/Qwen2.5-Coder-0.5B-Instruct', adapter_path, merge_path)

    llm = LLM(model=merge_path, dtype='float16', max_model_len=2048, gpu_memory_utilization=0.85)
    tokenizer = llm.get_tokenizer()
    params = SamplingParams(temperature=0.0, max_tokens=512)

    correct = 0
    for r in bird_dev:
        schema = get_schema(DATA_DIR, r['db_id'])
        messages = [
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': build_prompt(r['question'], schema, r.get('evidence', ''))},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        output = llm.generate([prompt], params)[0]
        pred_sql = output.outputs[0].text
        db_path = str(get_db_path(DATA_DIR, r['db_id']))
        if execution_match(pred_sql, r['gold_sql'], db_path):
            correct += 1

    print(f'EX on 300 dev: {correct}/{len(bird_dev)} = {correct/len(bird_dev):.4f}')
else:
    print('No successful adapter to evaluate')

## Full Training (30-50K samples)

Run with the best config from the grid search above.

In [ ]:
# Uncomment and run after selecting best config
# FULL_DATA = 'data/synsql_filtered_30000.jsonl'
# best_exp = experiments[0]  # replace with best from grid
# best_exp.name = f'{best_exp.name}_full30k'
# result = train(best_exp, FULL_DATA, str(RESULTS_DIR / best_exp.name))
# print(result)